In [ ]:
REPO_URL = "https://github.com/nikuradzedima/project_dl.git"
CHECKPOINT_URL = "https://huggingface.co/dimadmitrij734/project_dl/resolve/main/model_best.pth"
DATA_URL = "https://huggingface.co/dimadmitrij734/project_dl/resolve/main/demo_data.zip"

PROJECT_DIR = "/content/project_dl"
CHECKPOINT_PATH = "checkpoints/model_best.pth"
DATA_ZIP = "/content/demo_data.zip"
DATA_DIR = "/content/demo_data"
OUTPUT_DIR = "/content/demo_reconstructions"

## Клонируем репозиторий

In [ ]:
!rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

## Скачиваем лучшую модель

In [ ]:
!mkdir -p checkpoints
!wget -q -O {CHECKPOINT_PATH} {CHECKPOINT_URL}

## Данные для инференса

In [ ]:
!rm -rf {DATA_DIR} {DATA_ZIP}
!mkdir -p {DATA_DIR}
!wget -q -O {DATA_ZIP} {DATA_URL}
!unzip -q {DATA_ZIP} -d {DATA_DIR}

## Инференс

In [ ]:
!rm -rf {OUTPUT_DIR}
!python3 inference.py   inferencer.device=auto   inferencer.checkpoint_path={CHECKPOINT_PATH}   datasets.test.root={DATA_DIR}   inferencer.output_dir={OUTPUT_DIR}

## Визуализация

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

sample_ids = [path.stem for path in sorted(Path(OUTPUT_DIR).glob("*.png"))[:3]]
has_original = (Path(DATA_DIR) / "lensed").exists()
columns = 3 if has_original else 2
fig, axes = plt.subplots(len(sample_ids), columns, figsize=(4 * columns, 4 * len(sample_ids)), squeeze=False)

for row, image_id in enumerate(sample_ids):
    axes[row, 0].imshow(Image.open(Path(DATA_DIR) / "lensless" / f"{image_id}.png"))
    axes[row, 0].set_title("lensless")
    axes[row, 1].imshow(Image.open(Path(OUTPUT_DIR) / f"{image_id}.png"))
    axes[row, 1].set_title("reconstruction")
    if has_original:
        axes[row, 2].imshow(Image.open(Path(DATA_DIR) / "lensed" / f"{image_id}.png"))
        axes[row, 2].set_title("original")
    for column in range(columns):
        axes[row, column].axis("off")

plt.tight_layout()

## Метрики, если есть оригинал

In [ ]:
if has_original:
    !python3 calculate_metrics.py data_dir={DATA_DIR} prediction_dir={OUTPUT_DIR} device=cpu
